# Figuras, tabelas e números do artigo (E4 e E5)

Lê os experimentos gravados em `outputs/` (nunca roda algoritmo pesado) e gera:
- figuras em `paper/figures/*.pdf` (F1, F3, F4, F5 e a figura de recursos; a F2 é desenhada em TikZ no próprio texto);
- versões desenhadas pelo próprio LaTeX, com a fonte do artigo: `paper/figures/fig_network.tex` (TikZ, rede de exemplo) e os dados de `paper/figures/fig_scaling.tex` (pgfplots), em `paper/figures/data/scaling_points.csv` e `paper/figures/data/scaling_fits.csv`;
- tabelas em `paper/tables/*.tex` (T0 a T5);
- `docs/resultados-entrega2.md`, com cada número citável no texto e o arquivo de origem.

Reexecutar do zero produz os mesmos arquivos (PDF sem data de criação).

## 1. Imports, experimentos e constantes

In [ ]:
import json
import math

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from qmfnip.grover import SearchSpace, search_space
from qmfnip.instances import generate_random, max_flow
from qmfnip.outputs import PROJECT_ROOT, latest_experiment, write_latex_table
from qmfnip.plots import (
    PAPER_STYLE,
    plot_amplification,
    plot_counts,
    plot_network,
    plot_scaling,
    write_network_tikz,
    write_scaling_data,
)
from qmfnip.solvers import MILPSolver

matplotlib.rcParams.update(PAPER_STYLE)

In [ ]:
E1 = latest_experiment(contains="durr_hoyer-random_order-enumeration__layered__n6-27__runs200")
E2 = latest_experiment(contains="dicke-truth_table-arithmetic")
E3 = latest_experiment(contains="grover_ideal-grover_fake_fez-grover_ibm")
MILP_RANDOM = latest_experiment(contains="milp__random__n10-5120")
MILP_LAYERED = latest_experiment(contains="milp__layered__n18-4098")
FIGURES = PROJECT_ROOT / "paper" / "figures"
TABLES = PROJECT_ROOT / "paper" / "tables"
RESULTS_DOC = PROJECT_ROOT / "docs" / "resultados-entrega2.md"
PDF_METADATA = {"CreationDate": None, "Creator": None, "Producer": None}
FIGURE_INSTANCE = ("layered-n11-m24-B2-s0", E1)
HYPERCUBE_SIZES = (5, 6, 7, 8, 10)
HYPERCUBE_SEEDS = (0, 1, 2)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
facts = []


def fact(key: str, value, source, note: str = "") -> None:
    facts.append({"key": key, "value": value, "source": str(source).replace(str(PROJECT_ROOT) + "/", ""), "note": note})


def save(figure, name: str) -> None:
    figure.savefig(FIGURES / f"{name}.pdf", bbox_inches="tight", metadata=PDF_METADATA)
    plt.close(figure)


def fmt(value, digits: int = 3) -> str:
    return f"{value:,.{digits}f}".replace(",", "{,}")


{name: output.path.name for name, output in {"E1": E1, "E2": E2, "E3": E3, "MILP_RANDOM": MILP_RANDOM, "MILP_LAYERED": MILP_LAYERED}.items()}

## 2. Execução: o único cálculo novo é a T0 (viáveis × hipercubo)

Conta |F| = |{x : c·x ≤ B}| (programação dinâmica, sem enumerar) em instâncias aleatórias com custos de 1 a 5 e orçamento calibrado em 50% do corte que zera o fluxo, e confere a degenerescência com o MILP. Motiva a decisão D1 (§4 do artigo).

In [ ]:
rows = []
for size in HYPERCUBE_SIZES:
    for seed in HYPERCUBE_SEEDS:
        instance = generate_random(size, seed, budget_fraction=0.5, arc_factor=2.5)
        milp = MILPSolver().solve(instance, seed=0, time_limit=60)
        feasible = SearchSpace.count_feasible(instance)
        rows.append(
            {
                "instance_id": instance.instance_id, "n": instance.n, "m": instance.m, "B": instance.budget,
                "feasible": feasible, "hypercube": 2**instance.m, "sqrt_hypercube": math.sqrt(2**instance.m),
                "non_degenerate": bool(milp.certified_optimal and 0 < milp.objective < max_flow(instance)),
            }
        )
t0 = pd.DataFrame(rows)
t0["feasible_below_sqrt"] = t0["feasible"] < t0["sqrt_hypercube"]
t0

## 3. Figuras, tabelas e números

### E1: escala (F3, T1, T2) e contexto clássico (T5)

In [ ]:
per_instance = pd.read_csv(E1.path / "per_instance.csv")
fits = pd.read_csv(E1.path / "t2_scaling_fits.csv")
t1_raw = pd.read_csv(E1.path / "t1_instances.csv")
h4 = json.loads((E1.path / "h4.json").read_text())
budget = json.loads((E1.path / "query_budget_summary.json").read_text())
acceptance = json.loads((E1.path / "acceptance.json").read_text())
labels = {
    "durr_hoyer": "Dürr–Høyer (queries + evaluations)",
    "random_order": "random order (evaluations)",
    "enumeration": "exhaustive enumeration (evaluations)",
}
order = ["durr_hoyer", "random_order", "enumeration"]
per_instance["solver"] = pd.Categorical(per_instance["solver"], order)
per_instance = per_instance.sort_values(["solver", "N"])
figure = plot_scaling(
    per_instance, "N", "mean_cost", fits=fits[fits["x"] == "N"], labels=labels,
    xlabel="feasible interdiction plans $N = |\\mathcal{F}_k|$", ylabel="mean cost until the optimum is measured",
)
save(figure, "f3_scaling")
write_scaling_data(per_instance, "N", "mean_cost", fits[fits["x"] == "N"], FIGURES / "data")
for key, value in acceptance.items():
    fact(f"e1_acceptance_{key}", value, E1.path / "acceptance.json")
for key in ("beta", "ci_low", "ci_high", "verdict"):
    fact(f"h4_{key}", h4[key], E1.path / "h4.json")
for key, value in budget.items():
    fact(f"e1_{key}", value, E1.path / "query_budget_summary.json")
fits.round(4)

In [ ]:
t1_raw["width"] = t1_raw["n_nodes"].map(lambda n: int(round(math.sqrt(n - 2))))
grouped = t1_raw.sort_values("instance_seed").groupby(["width", "n_nodes", "n_arcs", "k", "kappa", "N"], as_index=False).agg(
    M=("M", lambda values: "/".join(map(str, values))),
    flow=("flow_without_interdiction", lambda values: "/".join(map(str, values))),
    optimum=("optimal_flow", lambda values: "/".join(str(int(value)) for value in values)),
)
grouped = grouped.sort_values("N")
grouped["N"] = grouped["N"].map(lambda value: f"{value:,}".replace(",", "{,}"))
write_latex_table(
    grouped[["width", "n_nodes", "n_arcs", "k", "kappa", "N", "M", "flow", "optimum"]],
    TABLES / "t1_instances.tex",
    caption="Instances of the scaling experiment (layered family, unit costs, $k<\\kappa$). Values separated by slashes refer to seeds 0/1/2.",
    label="tab:instances",
    wide=True,
    column_format="rrrrrrlll",
    headers=["$w$", "$n$", "$m$", "$k$", "$\\kappa$", "$N=|\\mathcal{F}_k|$", "$M$", "$F(0)$", "$F^{*}$"],
)
fact("e1_instances", len(t1_raw), E1.path / "t1_instances.csv")
fact("e1_N_min", int(t1_raw["N"].min()), E1.path / "t1_instances.csv")
fact("e1_N_max", int(t1_raw["N"].max()), E1.path / "t1_instances.csv")
fact("e1_M_max", int(t1_raw["M"].max()), E1.path / "t1_instances.csv")
grouped

In [ ]:
names = {"durr_hoyer": "Dürr--Høyer", "random_order": "Random order", "enumeration": "Enumeration"}
t2 = fits.copy()
t2["method"] = t2["solver"].map(names)
t2["regressor"] = t2["x"].map({"N": "$N$", "N_over_M": "$N/M$"})
t2["beta_ci"] = t2.apply(lambda row: f"{row['beta']:.3f} [{row['ci_low']:.3f}, {row['ci_high']:.3f}]", axis=1)
t2["r2_fmt"] = t2["r2"].map(lambda value: f"{value:.3f}")
t2 = t2.sort_values(["x", "solver"], key=lambda column: column.map({"N": 0, "N_over_M": 1, "durr_hoyer": 0, "random_order": 1, "enumeration": 2}))
write_latex_table(
    t2[["method", "regressor", "beta_ci", "r2_fmt"]],
    TABLES / "t2_scaling_fits.tex",
    caption="Fitted scaling exponents over the 30 instances: least-squares slope of $\\log_{10}$(mean cost) against $\\log_{10}$ of the regressor, with 95\\% bootstrap confidence intervals (2{,}000 resamples of instances).",
    label="tab:fits",
    column_format="llrr",
    headers=["Method", "Reg.", "$\\beta$ [95\\% CI]", "$R^2$"],
)
for row in fits.itertuples():
    fact(f"beta_{row.solver}_vs_{row.x}", round(row.beta, 4), E1.path / "t2_scaling_fits.csv", f"CI [{row.ci_low:.4f}, {row.ci_high:.4f}], R2 {row.r2:.4f}")
dh = per_instance[per_instance["solver"] == "durr_hoyer"]
fact("dh_cost_over_sqrtN_min", round(float(dh["cost_over_sqrt_N"].min()), 3), E1.path / "per_instance.csv")
fact("dh_cost_over_sqrtN_max", round(float(dh["cost_over_sqrt_N"].max()), 3), E1.path / "per_instance.csv")
largest = per_instance[per_instance["N"] == per_instance["N"].max()].groupby("solver", observed=True)["mean_cost"].mean()
for solver, value in largest.items():
    fact(f"largest_N_mean_cost_{solver}", round(float(value), 1), E1.path / "per_instance.csv", "mean over the 3 seeds with the largest N")
ro = per_instance[per_instance["solver"] == "random_order"]
fact("random_order_over_theory_mean", round(float((ro["mean_cost"] / ro["random_order_theory"]).mean()), 3), E1.path / "per_instance.csv")
t2

In [ ]:
t5 = pd.read_csv(E1.path / "t5_classical_context.csv").sort_values("N")
t5_tex = pd.DataFrame(
    {
        "n": t5["n_nodes"], "m": t5["n_arcs"], "k": t5["k"],
        "N": t5["N"].map(lambda value: f"{value:,}".replace(",", "{,}")),
        "milp": (1000 * t5["milp_median_s"]).map(lambda value: f"{value:.1f}"),
        "enumeration": t5["enumeration_median_s"].map(lambda value: f"{value:.3f}" if value < 10 else f"{value:.1f}"),
    }
)
frontiers = []
for output in (MILP_RANDOM, MILP_LAYERED):
    for row in json.loads((output.path / "frontier.json").read_text()):
        frontiers.append({**row, "experiment": output.path.name})
frontiers = pd.DataFrame(frontiers)
notes = "; ".join(
    f"{row.family}, $B$ = {int(round(100 * row.budget_fraction))}\\%: " + f"{int(row.n_star_nodes):,}".replace(",", "{,}") + " nodes" + ("" if not row.reached_largest else " (largest tested)")
    for row in frontiers.sort_values(["family", "budget_fraction"]).itertuples()
)
write_latex_table(
    t5_tex,
    TABLES / "t5_classical_context.tex",
    caption="Classical context: median wall-clock time of the MILP (HiGHS) and of the exhaustive enumeration of $\\mathcal{F}_k$ (one maximum-flow computation per plan, single thread) on the instances of Table~\\ref{tab:instances}.",
    label="tab:classical",
    column_format="rrrrrr",
    headers=["$n$", "$m$", "$k$", "$N$", "MILP (ms)", "Enumeration (s)"],
    notes="Largest network size $n^{*}$ such that, at every tested size up to $n^{*}$, the MILP proved optimality on all three seeds within 20\\,s (random and layered families, general costs, budget $B$ as a fraction of the cost of the cheapest cut that disconnects $s$ from $t$): " + notes + ".",
)
fact("milp_time_max_ms_e1", round(1000 * float(t1_raw["milp_time_s"].max()), 1), E1.path / "t1_instances.csv")
fact("enumeration_time_max_s_e1", round(float(t1_raw["enumeration_time_s"].max()), 1), E1.path / "t1_instances.csv")
for row in frontiers.itertuples():
    fact(f"milp_nstar_{row.family}_{row.budget_fraction}", int(row.n_star_nodes), f"outputs/{row.experiment}/frontier.json", f"reached_largest={row.reached_largest}")
t5

### Figura F1: rede de exemplo

In [ ]:
name, source = FIGURE_INSTANCE
instance = source.load_instances()[name]
space = search_space(instance)
figure = plot_network(instance, space.interdictions[space.optimal_indices[0]])
save(figure, "f1_example_network")
write_network_tikz(instance, space.interdictions[space.optimal_indices[0]], FIGURES / "fig_network.tex")
fact("f1_instance", name, source.path / "instances" / f"{name}.json", f"F0={max_flow(instance)}, F*={space.optimum}, N={space.size}, M={space.optimal_count}")

### T0: viáveis × hipercubo

In [ ]:
t0_tex = pd.DataFrame(
    {
        "n": t0["n"], "m": t0["m"], "B": t0["B"],
        "feasible": t0["feasible"].map(lambda value: f"{value:,}".replace(",", "{,}")),
        "sqrt": t0["sqrt_hypercube"].map(lambda value: f"{value:,.0f}".replace(",", "{,}")),
        "hypercube": t0["hypercube"].map(lambda value: f"$2^{{{int(math.log2(value))}}}$"),
    }
)
write_latex_table(
    t0_tex,
    TABLES / "t0_feasible_vs_hypercube.tex",
    caption="Number of feasible interdiction plans $|\\{x : c^\\top x\\le B\\}|$ against the hypercube in random instances (costs in $\\{1,\\dots,5\\}$, budget equal to half the cost of the cheapest cut that disconnects $s$ from $t$, seeds 0--2).",
    label="tab:feasible-vs-hypercube",
    column_format="rrrrrr",
    headers=["$n$", "$m$", "$B$", "Feasible plans", "$\\sqrt{2^m}$", "$2^m$"],
)
t0.to_csv(FIGURES.parent / "tables" / "t0_feasible_vs_hypercube.csv", index=False)
fact("t0_feasible_below_sqrt", f"{int(t0['feasible_below_sqrt'].sum())}/{len(t0)}", "paper/tables/t0_feasible_vs_hypercube.csv")
above = t0[~t0["feasible_below_sqrt"]]
fact("t0_above_sqrt_cases", "; ".join(f"m={row.m}: {row.feasible} > {row.sqrt_hypercube:.0f}" for row in above.itertuples()), "paper/tables/t0_feasible_vs_hypercube.csv")
threshold_m = min(m for m in sorted(t0["m"].unique()) if t0.loc[t0["m"] >= m, "feasible_below_sqrt"].all())
fact("t0_min_m_all_below", int(threshold_m), "paper/tables/t0_feasible_vs_hypercube.csv", "every instance with at least this m has fewer feasible plans than sqrt(2^m)")
fact("t0_max_feasible_m25", int(t0[t0["m"] == t0["m"].max()]["feasible"].max()), "paper/tables/t0_feasible_vs_hypercube.csv", f"sqrt(2^m) = {t0['sqrt_hypercube'].max():.0f}")
fact("t0_non_degenerate", f"{int(t0['non_degenerate'].sum())}/{len(t0)}", "paper/tables/t0_feasible_vs_hypercube.csv")
largest_t0 = t0.sort_values("m").iloc[-1]
fact("t0_largest_m", f"m={int(largest_t0['m'])}, feasible={int(largest_t0['feasible'])}, sqrt={largest_t0['sqrt_hypercube']:.0f}", "paper/tables/t0_feasible_vs_hypercube.csv")

### E2: recursos (T3 e figura)

In [ ]:
resources = pd.read_csv(E2.path / "resources.csv")
analytic = pd.read_csv(E2.path / "t3_analytic.csv")
wide = resources.pivot_table(index=["instance_id", "n", "m", "k", "N"], columns=["circuit", "backend"], values=["logical_qubits", "two_qubit_gates", "circuit_depth"], aggfunc="first")
wide = wide.reset_index().sort_values(["m", "k"])
backend = [name for name in resources["backend"].unique() if name != "logical"][0]
t3 = pd.DataFrame(
    {
        "n": wide["n"], "m": wide["m"], "k": wide["k"], "N": wide["N"],
        "dicke": wide[("two_qubit_gates", "dicke_preparation", "logical")].astype(int).astype(str) + " / " + wide[("two_qubit_gates", "dicke_preparation", backend)].astype(int).astype(str),
        "table": wide[("two_qubit_gates", "truth_table_oracle", "logical")].astype(int).astype(str) + " / " + wide[("two_qubit_gates", "truth_table_oracle", backend)].astype(int).astype(str),
        "arith_q": wide[("logical_qubits", "arithmetic_oracle", "logical")].astype(int),
        "arith": wide[("two_qubit_gates", "arithmetic_oracle", "logical")].astype(int).map(lambda value: f"{value:,}".replace(",", "{,}")) + " / " + wide[("two_qubit_gates", "arithmetic_oracle", backend)].astype(int).map(lambda value: f"{value:,}".replace(",", "{,}")),
        "beats": wide["instance_id"].map(analytic.set_index("instance_id")["x_alpha_beats_enumeration"]).map({True: "yes", False: "no"}),
    }
)
write_latex_table(
    t3,
    TABLES / "t3_resources.tex",
    caption="Resources of the interface pieces: two-qubit gates, logical (basis \\texttt{cx}, \\texttt{u}) / transpiled for the IBM Heron model \\texttt{FakeFez} (optimisation level 1). The truth-table oracle marks the $M$ optimal plans. The last column states whether $2^{n-2}<N$, the condition under which searching the pairs $(x,\\alpha)$ can beat the enumeration of $\\mathcal{F}_k$.",
    label="tab:resources",
    wide=True,
    column_format="rrrrrrrrc",
    headers=["$n$", "$m$", "$k$", "$N$", "$A_k$", "Truth table", "Arith. qubits", "Arith. oracle", "$2^{n-2}<N$"],
)
with plt.rc_context(PAPER_STYLE):
    figure, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True)
    circuit_labels = {"dicke_preparation": "Dicke preparation $A_k$", "truth_table_oracle": "truth-table oracle", "arithmetic_oracle": "arithmetic oracle over $(x,\\alpha)$"}
    for axis, name in zip(axes, ("logical", backend)):
        for index, (circuit, group) in enumerate(resources[resources["backend"] == name].groupby("circuit", sort=False)):
            group = group.sort_values(["m", "k"])
            axis.scatter(group["m"] + 0.15 * (group["k"] - 1.5), group["two_qubit_gates"], marker=["o", "s", "^"][index], s=22, edgecolors="white", linewidths=0.6, label=circuit_labels[circuit])
        axis.set_yscale("log")
        axis.set_xlabel("arcs $m$")
        axis.set_title("logical" if name == "logical" else f"transpiled ({name})")
        axis.grid(alpha=0.3, which="major", lw=0.4)
    axes[0].set_ylabel("two-qubit gates")
    axes[1].legend(loc="lower right")
    figure.tight_layout()
    save(figure, "f6_resources")
arith = resources[(resources["circuit"] == "arithmetic_oracle")]
fact("e2_m_range", f"{int(resources['m'].min())}-{int(resources['m'].max())}", E2.path / "resources.csv")
fact("e2_arith_qubits_range", f"{int(arith['logical_qubits'].min())}-{int(arith['logical_qubits'].max())}", E2.path / "resources.csv")
fact("e2_arith_2q_logical_range", f"{int(arith[arith['backend'] == 'logical']['two_qubit_gates'].min())}-{int(arith[arith['backend'] == 'logical']['two_qubit_gates'].max())}", E2.path / "resources.csv")
fact("e2_arith_2q_fake_range", f"{int(arith[arith['backend'] == backend]['two_qubit_gates'].min())}-{int(arith[arith['backend'] == backend]['two_qubit_gates'].max())}", E2.path / "resources.csv")
dicke = resources[(resources["circuit"] == "dicke_preparation")]
fact("e2_dicke_2q_logical_range", f"{int(dicke[dicke['backend'] == 'logical']['two_qubit_gates'].min())}-{int(dicke[dicke['backend'] == 'logical']['two_qubit_gates'].max())}", E2.path / "resources.csv")
fact("e2_x_alpha_beats_enumeration", f"{int(analytic['x_alpha_beats_enumeration'].sum())}/{len(analytic)}", E2.path / "t3_analytic.csv")
t3

### E3: amplificação (F4, F5, T4)

In [ ]:
theory = pd.read_csv(E3.path / "theory.csv")
amplification = pd.read_csv(E3.path / "amplification.csv")
h5 = json.loads((E3.path / "h5.json").read_text())
manifest = json.loads((E3.path / "manifest.json").read_text())
optimum = manifest["config"]["optimum"]
shots = manifest["config"]["shots"]
titles = {"interface": "interface: Dicke preparation over $\\mathcal{F}_1$ ($N=4$)", "hypercube": "hypercube: $H^{\\otimes 3}$ ($N=8$)"}
figure = plot_amplification(amplification, theory, shots, titles=titles)
save(figure, "f4_amplification")
best = {variant: int(group.loc[group["probability"].idxmax(), "iterations"]) for variant, group in theory.groupby("variant", sort=False)}
count_frames = []
for environment, key in (("ideal simulator", "ideal"), ("noisy simulator (FakeFez)", "fake_fez"), ("IBM hardware", "ibm")):
    path = E3.path / f"counts_{key}.csv"
    if path.exists():
        count_frames.append(pd.read_csv(path, dtype={"state": str}).assign(environment=environment))
counts = pd.concat(count_frames, ignore_index=True)
panels = pd.concat([counts[(counts["variant"] == variant) & (counts["iterations"] == iterations)].assign(panel=variant) for variant, iterations in best.items()], ignore_index=True)
feasible = {state for state in panels["state"].unique() if state.count("1") <= 1}
figure = plot_counts(panels, optimum, feasible, titles={key: f"{titles[key]}, $j={value}$" for key, value in best.items()})
save(figure, "f5_counts")
t4 = pd.read_csv(E3.path / "t4_hardware.csv")
t4_tex = pd.DataFrame(
    {
        "environment": t4["environment"]
        .str.replace("noisy simulator (FakeFez)", "noisy sim. (\\texttt{FakeFez})", regex=False)
        .where(t4["environment"] != "IBM hardware", "hardware (\\texttt{" + t4["backend"].str.replace("_", "\\_") + "})"),
        "variant": t4["variant"], "j": t4["iterations"],
        "depth": t4["depth"].where(t4["environment"] != "ideal simulator", "--"),
        "2q": t4["two_qubit_gates"].where(t4["environment"] != "ideal simulator", "--"),
        "theory": t4["theory"].map(lambda value: f"{value:.3f}"),
        "freq": t4["frequency"].map(lambda value: f"{value:.3f}"),
        "uniform": t4["uniform"].map(lambda value: f"{value:.3f}"),
        "p": t4["p_value"].map(lambda value: "$<10^{-300}$" if value == 0 else f"{value:.1e}"),
    }
)
write_latex_table(
    t4_tex,
    TABLES / "t4_hardware.tex",
    caption=f"Amplification at the optimal number of Grover iterations $j$ ({shots:,} shots per circuit)".replace(",", "{,}") + ". Depth and two-qubit gates after transpilation; $p$: one-sided binomial test of the frequency of the optimum against $1/N$.",
    label="tab:hardware",
    wide=True,
    column_format="llrrrrrrr",
    headers=["Environment", "Variant", "$j$", "Depth", "2q gates", "Theory", "Frequency", "$1/N$", "$p$"],
)
for row in amplification.itertuples():
    fact(f"e3_{row.environment}_{row.variant}_j{row.iterations}_frequency", round(row.frequency, 4), E3.path / "amplification.csv", f"theory {row.probability:.4f}, p={row.p_value:.2e}")
fact("h5_verdict", h5["verdict"], E3.path / "h5.json")
for row in amplification[amplification["variant"] == "interface"].itertuples():
    fact(f"e3_{row.environment}_interface_j{row.iterations}_feasible_frequency", round(row.feasible_frequency, 4), E3.path / "amplification.csv", "fraction of shots inside F_1")
fact("e3_instance", list(manifest["instance"].keys())[0], E3.path / "manifest.json", f"optimum {optimum}")
for row in t4[t4["environment"] == "IBM hardware"].head(1).itertuples():
    fact("e3_ibm_backend", row.backend, E3.path / "t4_hardware.csv", f"job {row.job_id}, finished {row.date}")
    fact("e3_ibm_qpu_seconds", row.qpu_seconds, E3.path / "t4_hardware.csv", "qpu_charge_time_seconds reported by IBM for the whole job")
    fact("e3_ibm_queue_minutes", round(row.queue_seconds / 60, 1), E3.path / "t4_hardware.csv", "from creation to start of execution")
    fact("e3_ibm_total_minutes", round(row.total_seconds / 60, 1), E3.path / "t4_hardware.csv", "from creation to finish")
t4_tex

Comparação exploratória do hardware com a probabilidade teórica e com o `FakeFez` (intervalo de Wilson e teste exato de Fisher). Ela foi definida depois da execução no `ibm_fez` e não faz parte do protocolo registrado.

In [ ]:
exploratory = []
for variant, iterations in best.items():
    rows = amplification[(amplification["variant"] == variant) & (amplification["iterations"] == iterations)].set_index("environment")
    hardware, noisy = rows.loc["IBM hardware"], rows.loc["noisy simulator (FakeFez)"]
    hits, noisy_hits = int(hardware["count"]), int(noisy["count"])
    interval = stats.binomtest(hits, shots).proportion_ci(0.95, method="wilson")
    fisher = stats.fisher_exact([[hits, shots - hits], [noisy_hits, shots - noisy_hits]]).pvalue
    exploratory.append(
        {
            "variant": variant,
            "iterations": iterations,
            "hardware": hardware["frequency"],
            "fake_fez": noisy["frequency"],
            "theory": hardware["probability"],
            "ratio_to_theory": hardware["frequency"] / hardware["probability"],
            "wilson_low": interval.low,
            "wilson_high": interval.high,
            "difference_to_fake_fez": hardware["frequency"] - noisy["frequency"],
            "fisher_p": fisher,
        }
    )
exploratory = pd.DataFrame(exploratory)
for row in exploratory.itertuples():
    for column in ("ratio_to_theory", "wilson_low", "wilson_high", "difference_to_fake_fez"):
        fact(f"e3_exploratory_{row.variant}_{column}", round(float(getattr(row, column)), 4), E3.path / "amplification.csv", "defined after the hardware run")
    fact(f"e3_exploratory_{row.variant}_fisher_p", f"{row.fisher_p:.2e}", E3.path / "amplification.csv", "two-sided Fisher exact test, hardware vs FakeFez; defined after the hardware run")
exploratory

### `docs/resultados-entrega2.md`

In [ ]:
experiments = {"E1": E1, "E2": E2, "E3": E3, "MILP (random)": MILP_RANDOM, "MILP (layered)": MILP_LAYERED}
lines = [
    "# Resultados da entrega 2 (gerado pelo notebook `05_article_figures`)",
    "",
    "Não editar à mão: reexecutar o notebook. Todo número citado no artigo sai desta lista.",
    "",
    "## Experimentos de origem",
    "",
    *[f"- **{name}:** `outputs/{output.path.name}`" for name, output in experiments.items()],
    "",
    "## Números",
    "",
    "| Chave | Valor | Origem | Nota |",
    "|---|---|---|---|",
    *[f"| `{item['key']}` | {item['value']} | `{item['source']}` | {item['note']} |" for item in facts],
    "",
]
RESULTS_DOC.write_text("\n".join(lines))
len(facts)